# 13 - A simple C# hosted agent with Microsoft Agent Framework

## Required permissions

- **You:** **Foundry Owner on the parent Foundry resource**, covering workshop hosted-agent authoring/invocation, and `az login` in the deployment tenant. Project-only Foundry User is not the full authoring setup.
- **Existing service resources:** a Foundry project, chat deployment and supported hosted **.NET code deployment / remote build** in its region. The agent's runtime identity needs model access. This notebook assigns no Azure roles; no Search, Blob, SharePoint, broker app or tenant administrator permission is required.

**Resource prefix:** set `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Azure name: `alice01-hosted-agent`, matching Python13. C# source folder: `alice01-csharp-hosted-agent`, so Python's `main.py` and `requirements.txt` are untouched. The same Azure name means another immutable version, not a separate C# agent; use different prefixes to keep both language versions independently. Aliases prevent collisions, not enforce access isolation.

A **hosted agent** is your own code running as a managed application in Foundry. This notebook writes and deploys **C# agent code**, not a Python agent called from C#.

```text
C# source + .csproj --zip--> Foundry remote build + hosted agent <--Responses API-- this notebook
   AIAgent + 1 function tool
   AgentHost / Foundry Responses
```

You will write the agent, package its source, upload one immutable version, wait for provisioning, make two example calls and optionally delete it. No Python runtime, Docker, ACR or `azd` is required by this path. The deployed function returns local workshop tips; it does not query Web IQ or Search.

**C# track:** this optional notebook uses deprecated Polyglot/.NET Interactive tooling. The [.NET 10 console app and step-by-step README](../c-sharp-console-apps/13-hosted-agent/README.md) are the supported alternative. All lab helpers are inline below; no imports from another notebook or console project. The generated hosted application uses NuGet packages, restored by Foundry. Hosted compute, sessions and model inference can incur charges; use synthetic questions and clear outputs before sharing.

## Prerequisites and configuration

Use an existing project with hosted .NET remote-build support and a chat deployment; Python hosting availability does not guarantee .NET hosting availability. Install Azure CLI and the .NET 10 SDK. Sign in with `az login --tenant <AZURE_TENANT_ID>`.

Set the kernel working directory to **`c-sharp-notebooks`**. `Path.GetFullPath("..")` resolves the workshop root. Configuration loads root `.env`, then `.env.azure`; both override environment values, and `.env.azure` wins.

Required: `WORKSHOP_PREFIX`, `AZURE_TENANT_ID`, `AZURE_AI_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`. The endpoint check accepts public-cloud `https://<account>.services.ai.azure.com/api/projects/<project>` only, without credentials/query/fragment. Use a model **deployment name**.

The setup cell defines the standalone BCL helpers. It does not authenticate, write files or contact Azure. `Azure CLI` provides your Foundry-audience token for later requests; inside the hosted application, `DefaultAzureCredential` uses the platform-provided agent credential. No keys or user tokens are uploaded.

The host/tool code follows the [official C# local-tools sample](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/csharp/hosted-agents/agent-framework/local-tools) and [Agent Framework Foundry hosting guide](https://learn.microsoft.com/agent-framework/hosting/foundry-hosted-agent?pivots=programming-language-csharp).

In [ ]:
#nullable enable
using System;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using System.Threading.Tasks;
using System.Text.RegularExpressions;
using System.IO.Compression;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Security.Cryptography;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;

public sealed record HostedSettings(string Tenant, string ProjectEndpoint, string Model, string Prefix)
{
    public string AgentName => Prefix + "-hosted-agent";
    public string Folder(string root) => Path.Combine(Path.GetFullPath(root), Prefix + "-csharp-hosted-agent");

    public static HostedSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". See .env.example.");
        var settings = new HostedSettings(Required("AZURE_TENANT_ID"),
            Required("AZURE_AI_PROJECT_ENDPOINT").TrimEnd('/'),
            Required("AZURE_AI_MODEL_DEPLOYMENT_NAME"), Required("WORKSHOP_PREFIX"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _))
            throw new InvalidOperationException("AZURE_TENANT_ID must be a tenant GUID.");
        if (!Regex.IsMatch(Prefix, @"\A[a-z][a-z0-9]{2,19}\z"))
            throw new InvalidOperationException("Use 3-20 lowercase letters/digits for WORKSHOP_PREFIX, starting with a letter.");
        if (!Regex.IsMatch(ProjectEndpoint,
            @"\Ahttps://[a-z0-9][a-z0-9-]*\.services\.ai\.azure\.com/api/projects/[a-zA-Z0-9._-]+\z"))
            throw new InvalidOperationException("Use a public-cloud HTTPS Foundry project endpoint, without a query, fragment or credentials.");
        if (string.IsNullOrWhiteSpace(Model) || Model.Any(char.IsControl))
            throw new InvalidOperationException("AZURE_AI_MODEL_DEPLOYMENT_NAME must be a non-empty deployment name.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string raw in File.ReadLines(path))
        {
            string line = raw.Trim();
            if (line.StartsWith("export ", StringComparison.Ordinal)) line = line.Substring(7).TrimStart();
            if (line.Length == 0 || line.StartsWith("#", StringComparison.Ordinal)) continue;
            int equals = line.IndexOf('=');
            if (equals <= 0) continue;
            string name = line.Substring(0, equals).Trim();
            if (!Regex.IsMatch(name, @"\A[A-Za-z_][A-Za-z0-9_]*\z")) continue;
            string value = line.Substring(equals + 1).Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[value.Length - 1] == '"') ||
                (value[0] == '\'' && value[value.Length - 1] == '\'')))
                value = value.Substring(1, value.Length - 2);
            Environment.SetEnvironmentVariable(name, value);
        }
    }

    public async Task<string> GetTokenAsync()
    {
        Validate();
        const string command = "az account get-access-token --resource https://ai.azure.com " +
            "--query accessToken --output tsv --only-show-errors --tenant ";
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? Environment.GetEnvironmentVariable("ComSpec") ?? "cmd.exe" : "az",
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            UseShellExecute = false,
            CreateNoWindow = true,
        };
        if (OperatingSystem.IsWindows())
        {
            start.ArgumentList.Add("/d");
            start.ArgumentList.Add("/c");
            start.ArgumentList.Add(command + Tenant);
        }
        else
        {
            foreach (string argument in (command.Substring(3) + Tenant).Split(' '))
                start.ArgumentList.Add(argument);
        }
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> error = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(120));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            throw new InvalidOperationException("Azure CLI token acquisition timed out.");
        }
        string token = (await output).Trim();
        await error;
        if (process.ExitCode != 0 || token.Length == 0 || token.Any(char.IsWhiteSpace))
            throw new InvalidOperationException("Azure CLI could not obtain a token. Sign in with az login in AZURE_TENANT_ID.");
        return token;
    }
}

public static class HostedSources
{
    public const string Project = """
        <Project Sdk="Microsoft.NET.Sdk.Web">
          <PropertyGroup>
            <TargetFramework>net10.0</TargetFramework>
            <AssemblyName>WorkshopAgent</AssemblyName>
            <ImplicitUsings>enable</ImplicitUsings>
            <Nullable>enable</Nullable>
          </PropertyGroup>
          <ItemGroup>
            <PackageReference Include="Microsoft.Agents.AI.Foundry.Hosting" Version="1.22.0-preview.260918.1" />
            <PackageReference Include="Azure.AI.Projects" Version="3.0.0-beta.2" />
          </ItemGroup>
        </Project>
        """;

    public const string Program = """
        using Azure.AI.AgentServer.Core;
        using Azure.AI.Projects;
        using Azure.Identity;
        using Microsoft.Agents.AI;
        using Microsoft.Agents.AI.Foundry.Hosting;
        using Microsoft.Extensions.AI;

        var projectEndpoint = new Uri(Environment.GetEnvironmentVariable("FOUNDRY_PROJECT_ENDPOINT")
            ?? throw new InvalidOperationException("FOUNDRY_PROJECT_ENDPOINT is not set."));
        var deployment = Environment.GetEnvironmentVariable("AZURE_AI_MODEL_DEPLOYMENT_NAME")
            ?? throw new InvalidOperationException("AZURE_AI_MODEL_DEPLOYMENT_NAME is not set.");

        AIAgent agent = new AIProjectClient(projectEndpoint, new DefaultAzureCredential())
            .AsAIAgent(
                model: deployment,
                instructions: "You are a friendly workshop assistant. Use get_workshop_tip when asked about a workshop topic.",
                name: "workshop-assistant",
                tools: [
                    AIFunctionFactory.Create(WorkshopTips.Get, "get_workshop_tip",
                        "Return a one-line tip about a workshop topic such as web iq, foundry iq or indexer.")
                ]);

        var builder = AgentHost.CreateBuilder(args);
        builder.Services.AddFoundryResponses(agent);
        builder.RegisterProtocol("responses", endpoints => endpoints.MapFoundryResponses());
        var app = builder.Build();
        app.Run();
        """;

    public const string Tips = """
        public static class WorkshopTips
        {
            private static readonly Dictionary<string, string> Tips = new(StringComparer.OrdinalIgnoreCase)
            {
                ["web iq"] = "Web IQ grounds answers in the public web through one API key.",
                ["foundry iq"] = "Foundry IQ puts several knowledge sources behind one knowledge base.",
                ["indexer"] = "An indexer pulls documents from a data source into an AI Search index.",
            };

            public static string Get(string topic) => Tips.TryGetValue(topic, out var tip)
                ? tip : "No tip for that topic. Try: " + string.Join(", ", Tips.Keys);
        }
        """;
}

public sealed class HostedWorkshop
{
    public const string Question = "Give me a tip about foundry iq.";
    public const string ToolQuestion = "What is the workshop tip for indexer? Quote it exactly.";
    // Match the current azure-ai-projects allow_preview agent-operation header.
    public const string Features = "A2AAgents=V1Preview,WorkflowAgents=V1Preview,DraftAgents=V1Preview," +
        "VoiceAgents=V1Preview,DigitalWorker=V1Preview,GitHubCopilot=V1Preview,Skills=V1Preview," +
        "MicrosoftCopilot=V1Preview,ModelRouterControls=V1Preview";
    private static readonly string[] SourceFiles = { "WorkshopAgent.csproj", "Program.cs", "WorkshopTips.cs" };
    private readonly HostedSettings settings;
    private readonly HttpClient client;
    private readonly Func<Task<string>> getToken;
    private readonly Func<DateTimeOffset> utcNow;
    private readonly Func<TimeSpan, Task> delay;

    public HostedWorkshop(HostedSettings settings, HttpClient client, Func<Task<string>>? getToken = null,
        Func<DateTimeOffset>? utcNow = null, Func<TimeSpan, Task>? delay = null)
    {
        settings.Validate();
        this.settings = settings;
        this.client = client;
        this.getToken = getToken ?? settings.GetTokenAsync;
        this.utcNow = utcNow ?? (() => DateTimeOffset.UtcNow);
        this.delay = delay ?? (duration => Task.Delay(duration));
    }

    public void WriteFiles(string root)
    {
        string folder = settings.Folder(root);
        Directory.CreateDirectory(folder);
        RejectLinkedPath(folder);
        foreach (string name in SourceFiles) RejectLinkedPath(Path.Combine(folder, name));
        File.WriteAllText(Path.Combine(folder, SourceFiles[0]), HostedSources.Project + Environment.NewLine);
        File.WriteAllText(Path.Combine(folder, SourceFiles[1]), HostedSources.Program + Environment.NewLine);
        File.WriteAllText(Path.Combine(folder, SourceFiles[2]), HostedSources.Tips + Environment.NewLine);
        Console.WriteLine("Wrote C# hosted-agent source to " + folder);
    }

    public string Package(string root)
    {
        string folder = settings.Folder(root);
        RejectLinkedPath(folder);
        foreach (string name in SourceFiles)
        {
            string path = Path.Combine(folder, name);
            RejectLinkedPath(path);
            if (!File.Exists(path))
                throw new InvalidOperationException("Missing " + name + ". Run write-agent first.");
        }
        string zip = Path.Combine(folder, "agent.zip");
        RejectLinkedPath(zip);
        using (var stream = new FileStream(zip, FileMode.Create, FileAccess.Write))
        using (var bundle = new ZipArchive(stream, ZipArchiveMode.Create))
            foreach (string name in SourceFiles)
                bundle.CreateEntryFromFile(Path.Combine(folder, name), name, CompressionLevel.Optimal);
        Console.WriteLine("Packaged exactly the three C# source files: " + zip);
        return zip;
    }

    public JsonObject CodeMetadata() => new JsonObject
    {
        ["definition"] = new JsonObject
        {
            ["kind"] = "hosted",
            ["cpu"] = "0.5",
            ["memory"] = "1Gi",
            ["protocol_versions"] = new JsonArray(new JsonObject { ["protocol"] = "responses", ["version"] = "2.0.0" }),
            ["environment_variables"] = new JsonObject { ["AZURE_AI_MODEL_DEPLOYMENT_NAME"] = settings.Model },
            ["code_configuration"] = new JsonObject
            {
                ["runtime"] = "dotnet_10",
                ["entry_point"] = new JsonArray("dotnet", "WorkshopAgent.dll"),
                ["dependency_resolution"] = "remote_build",
            },
        },
        ["description"] = "Workshop hosted C# agent with one function tool",
        ["metadata"] = new JsonObject { ["workshop"] = "iq", ["attendee"] = settings.Prefix },
    };

    public async Task<JsonObject> DeployAsync(string root)
    {
        string zip = Path.Combine(settings.Folder(root), "agent.zip");
        RejectLinkedPath(settings.Folder(root));
        RejectLinkedPath(zip);
        if (!File.Exists(zip)) throw new InvalidOperationException("Missing agent.zip. Run package first.");
        using var stream = File.OpenRead(zip);
        if (stream.Length == 0 || stream.Length > 250L * 1024 * 1024)
            throw new InvalidOperationException("Code zip must be non-empty and no larger than 250 MB.");
        using (var bundle = new ZipArchive(stream, ZipArchiveMode.Read, leaveOpen: true))
        {
            if (!bundle.Entries.Select(e => e.FullName).OrderBy(n => n).SequenceEqual(SourceFiles.OrderBy(n => n)))
                throw new InvalidOperationException("Code zip must contain only the three expected source files. Run package again.");
        }
        stream.Position = 0;
        string digest = Convert.ToHexString(SHA256.HashData(stream)).ToLowerInvariant();
        stream.Position = 0;
        using var content = new MultipartFormDataContent();
        content.Add(new StringContent(CodeMetadata().ToJsonString(), Encoding.UTF8, "application/json"), "metadata");
        var code = new StreamContent(stream);
        code.Headers.ContentType = new MediaTypeHeaderValue("application/zip");
        content.Add(code, "code", "agent.zip");
        using var request = new HttpRequestMessage(HttpMethod.Post, Url("/versions")) { Content = content };
        request.Headers.Add("x-ms-code-zip-sha256", digest);
        JsonObject version = await SendAsync(request);
        ValidateVersion(version);
        Console.WriteLine("Created version " + version["version"] + " status: " + version["status"]);
        return version;
    }

    public async Task<JsonObject> LatestAsync()
    {
        using var request = new HttpRequestMessage(HttpMethod.Get, Url(""));
        JsonObject agent = await SendAsync(request);
        if ((string?)agent["name"] != settings.AgentName || agent["versions"]?["latest"] is not JsonObject version)
            throw new InvalidOperationException("Agent lookup did not return this prefix's latest version.");
        ValidateVersion(version);
        return version;
    }

    public async Task<JsonObject> StatusAsync(string versionId)
    {
        ValidateVersionId(versionId);
        using var request = new HttpRequestMessage(HttpMethod.Get, Url("/versions/" + Uri.EscapeDataString(versionId)));
        JsonObject version = await SendAsync(request);
        ValidateVersion(version);
        if ((string?)version["version"] != versionId)
            throw new InvalidOperationException("Foundry returned a different version than requested.");
        Console.WriteLine("Version " + versionId + " status: " + version["status"]);
        return version;
    }

    public async Task<JsonObject> WaitAsync(string versionId, TimeSpan? timeout = null)
    {
        ValidateVersionId(versionId);
        TimeSpan budget = timeout ?? TimeSpan.FromMinutes(15);
        if (budget <= TimeSpan.Zero || budget > TimeSpan.FromMinutes(15))
            throw new ArgumentException("Wait timeout must be greater than zero and at most 15 minutes.");
        DateTimeOffset deadline = utcNow() + budget;
        while (true)
        {
            TimeSpan remaining = deadline - utcNow();
            if (remaining <= TimeSpan.Zero)
                throw new TimeoutException("Agent version is still not active after the wait timeout. Run wait again; do not redeploy.");
            using var request = new HttpRequestMessage(HttpMethod.Get, Url("/versions/" + Uri.EscapeDataString(versionId)));
            JsonObject version = await SendAsync(request, remaining);
            ValidateVersion(version);
            if ((string?)version["version"] != versionId)
                throw new InvalidOperationException("Foundry returned a different version than requested.");
            string status = ((string?)version["status"] ?? "").ToLowerInvariant();
            Console.WriteLine("Version " + versionId + " status: " + status);
            if (status == "active") return version;
            if (status is "failed" or "deleted" or "deleting")
                throw new InvalidOperationException("Agent provisioning ended with status " + status + ". Inspect Foundry build/runtime diagnostics.");
            remaining = deadline - utcNow();
            if (remaining > TimeSpan.Zero)
                await delay(remaining < TimeSpan.FromSeconds(15) ? remaining : TimeSpan.FromSeconds(15));
        }
    }

    public async Task<JsonObject> InvokeAsync(string question, bool showOutputTypes = false)
    {
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        using var request = new HttpRequestMessage(HttpMethod.Post,
            Url("/endpoint/protocols/openai/responses"))
        {
            Content = new StringContent(new JsonObject { ["input"] = question, ["store"] = false }.ToJsonString(),
                Encoding.UTF8, "application/json"),
        };
        JsonObject response = await SendAsync(request);
        if ((string?)response["status"] != "completed" || response["error"] != null ||
            response["incomplete_details"] != null)
            throw new InvalidOperationException("Hosted response did not complete. No automatic question retry was made.");
        if (response["output"] is not JsonArray output)
            throw new InvalidOperationException("Hosted response omitted its output array.");
        string answer = string.Join(Environment.NewLine, output.OfType<JsonObject>()
            .Where(item => (string?)item["type"] == "message")
            .SelectMany(item => (item["content"] as JsonArray ?? new JsonArray()).OfType<JsonObject>())
            .Where(item => (string?)item["type"] == "output_text").Select(item => (string?)item["text"] ?? ""));
        if (string.IsNullOrWhiteSpace(answer)) throw new InvalidOperationException("Hosted response contained no answer text.");
        Console.WriteLine(SafeText(answer, 16000));
        Console.WriteLine("Session: " + SafeText((string?)response["agent_session_id"] ?? "(not returned)", 300));
        if (showOutputTypes)
        {
            foreach (JsonObject item in output.OfType<JsonObject>())
                Console.WriteLine("output item: " + SafeText((string?)item["type"] ?? "(unknown)", 100));
            Console.WriteLine("Review the tool-call output/logs. An exact-looking answer alone does not prove a function call.");
        }
        return response;
    }

    public async Task CleanupAsync(bool confirm)
    {
        if (!confirm) throw new ArgumentException("Cleanup requires exactly: cleanup --confirm.");
        JsonObject latest = await LatestAsync();
        if ((string?)latest["metadata"]?["workshop"] != "iq" ||
            (string?)latest["metadata"]?["attendee"] != settings.Prefix)
            throw new InvalidOperationException("Refusing cleanup: latest version is not marked as this prefix's workshop agent.");
        using var request = new HttpRequestMessage(HttpMethod.Delete, Url(""));
        await SendAsync(request);
        Console.WriteLine("Deleted only " + settings.AgentName + " and its versions. Local source files and shared resources retained.");
    }

    public static void ValidateVersionId(string version)
    {
        if (!Regex.IsMatch(version, @"\A[A-Za-z0-9][A-Za-z0-9._-]{0,79}\z") || version is "." or "..")
            throw new ArgumentException("Use a valid Foundry version identifier, not a path or URL.");
    }

    private void ValidateVersion(JsonObject version)
    {
        if ((string?)version["name"] != settings.AgentName || (string?)version["definition"]?["kind"] != "hosted")
            throw new InvalidOperationException("Foundry did not return this prefix's hosted agent version.");
        ValidateVersionId((string?)version["version"] ?? "");
        if (string.IsNullOrWhiteSpace((string?)version["status"]))
            throw new InvalidOperationException("Foundry version omitted its provisioning status.");
    }

    private string Url(string suffix) => settings.ProjectEndpoint + "/agents/" +
        Uri.EscapeDataString(settings.AgentName) + suffix + "?api-version=v1";

    private async Task<JsonObject> SendAsync(HttpRequestMessage request, TimeSpan? timeout = null)
    {
        using var cancellation = new CancellationTokenSource(timeout ?? TimeSpan.FromMinutes(5));
        string token = await getToken().WaitAsync(cancellation.Token);
        if (string.IsNullOrWhiteSpace(token) || token.Any(char.IsWhiteSpace))
            throw new InvalidOperationException("Token acquisition returned an invalid bearer token.");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        request.Headers.Add("Foundry-Features", Features);
        request.Headers.Accept.Add(new MediaTypeWithQualityHeaderValue("application/json"));
        using HttpResponseMessage response = await client.SendAsync(request, cancellation.Token);
        string body = (await response.Content.ReadAsStringAsync(cancellation.Token)).Replace(token, "<redacted>");
        if (response.StatusCode != System.Net.HttpStatusCode.OK)
        {
            Console.Error.WriteLine("HTTP " + (int)response.StatusCode + " " + response.ReasonPhrase);
            Console.Error.WriteLine("UTC (Date): " + response.Headers.Date);
            foreach (string header in new[] { "x-ms-request-id", "apim-request-id", "x-ms-correlation-request-id" })
                if (response.Headers.TryGetValues(header, out var values))
                    Console.Error.WriteLine(header + ": " + SafeText(string.Join(", ", values).Replace(token, "<redacted>"), 300));
            Console.Error.WriteLine("Response body: " + SafeText(body, 4000));
            if (response.StatusCode == System.Net.HttpStatusCode.Conflict && request.Method == HttpMethod.Delete)
                Console.Error.WriteLine("Active sessions may block deletion. Close them in Foundry and rerun cleanup. This lab never force-deletes sessions.");
            throw new InvalidOperationException("Foundry request failed with HTTP " + (int)response.StatusCode + ". No automatic retry was made.");
        }
        try { return JsonNode.Parse(body) as JsonObject ?? throw new JsonException(); }
        catch (JsonException) { throw new InvalidOperationException("Foundry returned invalid JSON instead of an object."); }
    }

    private static void RejectLinkedPath(string path)
    {
        if ((File.Exists(path) || Directory.Exists(path)) && (File.GetAttributes(path) & FileAttributes.ReparsePoint) != 0)
            throw new InvalidOperationException("Refusing to read or overwrite a linked path: " + path);
    }

    private static string SafeText(string text, int max) =>
        new string(text.Where(c => !char.IsControl(c) || c is '\n' or '\r' or '\t').Take(max).ToArray());
}

string root = Path.GetFullPath("..");
HostedSettings settings = HostedSettings.Load(root);
HttpClient client = new HttpClient { Timeout = System.Threading.Timeout.InfiniteTimeSpan };
Func<Task<string>> tokenProvider = () => settings.GetTokenAsync();
Func<DateTimeOffset> clockProvider = () => DateTimeOffset.UtcNow;
Func<TimeSpan, Task> delayProvider = duration => Task.Delay(duration);
HostedWorkshop lab = new HostedWorkshop(settings, client, () => tokenProvider(),
    () => clockProvider(), duration => delayProvider(duration));
string selectedVersion = "";
JsonObject? created = null;
Console.WriteLine("Project: " + settings.ProjectEndpoint);
Console.WriteLine("Agent name: " + settings.AgentName);
Console.WriteLine("C# source folder: " + settings.Folder(root));


## 1. Write the C# agent

The next cell writes **three files** into your root `<prefix>-csharp-hosted-agent` folder:

| File | What it explains |
|---|---|
| `WorkshopAgent.csproj` | .NET 10 web application with pinned Foundry hosting/projects NuGet packages. |
| `WorkshopTips.cs` | A normal case-insensitive C# function returning the same three tips/fallback as Python13. |
| `Program.cs` | `AIFunctionFactory.Create` exposes `get_workshop_tip`; `AsAIAgent` combines the model, instructions and tool. `AgentHost`, `AddFoundryResponses` and `MapFoundryResponses` host your application on the Responses protocol. |

`FOUNDRY_PROJECT_ENDPOINT` and runtime identity come from Foundry; deployment supplies `AZURE_AI_MODEL_DEPLOYMENT_NAME`. Do not put user tokens, API keys or reserved `FOUNDRY_*` variables in the uploaded environment.

**This cell overwrites these generated source files.** Open them in your editor to inspect the real C# code. After editing a tip, rerun packaging/deployment/wait, not this cell. Writing makes no token request or Azure call.

Optional local compile check (replace the prefix): `dotnet build .\alice01-csharp-hosted-agent\WorkshopAgent.csproj`. Local validation needs NuGet access, but deployment does not require a local publish or container build.

In [ ]:
lab.WriteFiles(root);

## 2. Package the existing source

Zip exactly `WorkshopAgent.csproj`, `Program.cs` and `WorkshopTips.cs` at the archive root. This preserves your source edits and excludes `.env`, credentials, `bin`, `obj` and unrelated files. `package` is independent of source writing: missing files tell you to run section 1 rather than silently rewriting code.

The resulting `agent.zip` stays inside your generated C# folder. This cell creates no Azure resource and installs no packages.

In [ ]:
string zipPath = lab.Package(root);

## 3. Upload the zip and create an agent version

An **agent version** is an immutable snapshot. The upload is the C# REST equivalent of Python13's `create_version_from_code`: a multipart request with a JSON `metadata` part, binary `code` part and SHA-256 archive header. This cell reads the existing archive; it does not rewrite/package it or call the model.

| Setting | Meaning |
|---|---|
| `runtime` / `entry_point` | `dotnet_10`, started with `dotnet WorkshopAgent.dll`. |
| `dependency_resolution` | `remote_build`: Foundry restores/builds the uploaded .NET project. |
| `protocol_versions` | Responses `2.0.0`, matching Python13. |
| `cpu` / `memory` | `0.5` / `1Gi`. |
| `environment_variables` | Existing model deployment name only, no secrets. |

The definition is visible as `lab.CodeMetadata()` in the inline helper. The request uses the current Python SDK's `v1` agent endpoint and preview headers; Azure CLI requests the `https://ai.azure.com` resource, equivalent to `https://ai.azure.com/.default`.

After deployment, the returned ID is retained only in this kernel's `selectedVersion`. Publishing from either language under the same prefix creates another version of the same Azure agent. Unsupported region/runtime, authorization and package failures are not replaced by a different implementation.

In [ ]:
created = await lab.DeployAsync(root);
selectedVersion = (string)created["version"]!;

## 4. Wait until the version is active

Poll that **fixed version** every 15 seconds, for at most 15 minutes, and stop on failed/deleting/deleted. A timeout does not justify creating another version; rerun only this cell after checking Foundry diagnostics.

After a kernel restart, either set `selectedVersion` to the ID printed by deployment or let the cell read/select the **existing latest version**. It never uploads, rewrites files or calls the model. `status` is provisioning status, not proof that the application starts; a successful invocation below verifies runtime readiness.

In [ ]:
if (string.IsNullOrEmpty(selectedVersion))
{
    JsonObject existing = await lab.LatestAsync();
    selectedVersion = (string)existing["version"]!;
}
JsonObject activeVersion = await lab.WaitAsync(selectedVersion);
Console.WriteLine("Agent version is active: " + settings.AgentName + " version " + selectedVersion);

## 5. Call your hosted C# agent

This request goes to `<project>/agents/<prefix>-hosted-agent/endpoint/protocols/openai/responses?api-version=v1`, not the bare model endpoint. There is **no `model` argument**. It sends `input` and `store=false` once, without a prior-response/session selector; the first call can create a hosted session and return `agent_session_id`.

Invocation follows the **agent name's endpoint routing**, as Python13 does. Selecting a version for polling does not pin this call to it; check routing if the endpoint was customized elsewhere. An active version can still produce HTTP 424 for startup/session problems. HTTP and incomplete/failed response errors are raised, not shown as empty successes, and no automatic question retries are made.

`store=false` is not a guarantee of no platform telemetry/session retention. The notebook does not save tokens, prompts, answers or session IDs to disk.

In [ ]:
JsonObject response = await lab.InvokeAsync(HostedWorkshop.Question);

## 6. See the function tool at work

Ask once for the `indexer` tip; print the answer, session ID and output item types. Review tool-call output/runtime logs. A model may know the original indexer definition without executing your function, so a plausible or exact-looking answer **does not prove tool use**.

For a stronger experiment, change the generated `WorkshopTips.cs` tip to a unique synthetic sentence, rerun sections 2-4 and ask again. Do not rerun section 1, which overwrites that edit. Tool code changes require a new immutable agent version; existing sessions/routes may still use an older one. There is no Search, Web IQ or toolbox retrieval in this local function.

In [ ]:
JsonObject toolResponse = await lab.InvokeAsync(HostedWorkshop.ToolQuestion, showOutputTypes: true);

## Cleanup: opt-in, scoped and non-forcing

Hosted agents/sessions can incur charges or consume quota. Set `CLEAN_UP = true` only to delete **your** `<prefix>-hosted-agent` and its versions. The helper checks the latest hosted version's name and workshop/prefix metadata before deleting. It does not remove shared project/model resources or local source files, and it never force-deletes active sessions.

HTTP 409 can mean active sessions block deletion: close them in Foundry, then rerun cleanup explicitly. If you used both Python13 and C#13 with the same prefix, this agent name contains both languages' versions. The default below is no cleanup and no request.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
    await lab.CleanupAsync(confirm: true);
else
    Console.WriteLine("Cleanup skipped. Set CLEAN_UP = true to delete " + settings.AgentName);

In [ ]:
client.Dispose();